# Agents, Tool Use, and Agentic Frameworks
## Live Demo — Instructor-Run
**ML System Design and LLMOps | Week 4, Session 2**

The final session of the course. We've gone from basic ML systems to LLM serving to RAG. Today we look at agents — systems where an LLM doesn't just answer a single question, but takes a sequence of actions in the world.

The engineering challenge shifts: it's no longer about getting a single good response. It's about loop control, safety, cost, and reliability over multi-step execution.

Four things we'll build:
1. Function calling — how tools are defined and dispatched
2. ReAct loop simulation — the Thought→Action→Observation cycle
3. Loop detection — catching stuck agents before they run forever
4. Cost tracking — token budgets with graceful degradation

## Part 1: Function Calling
*Tools are just Python functions with JSON schemas*

We're going to define three tools, give them JSON schemas in the OpenAI function-calling format, and show how a dispatcher routes LLM tool calls to the right function. This is the plumbing underneath every agent framework.

In [ ]:
import json
import math
import ast

# ── Tool implementations ──────────────────────────────────────────────────────

def get_weather(city: str, unit: str = "celsius") -> dict:
    """Get current weather for a given city."""
    # Simulated weather data — in production: call a weather API
    simulated_weather = {
        "Mumbai":    {"temp_c": 32, "condition": "Humid and partly cloudy", "humidity": 85},
        "Bangalore": {"temp_c": 24, "condition": "Pleasant, light breeze",  "humidity": 60},
        "Delhi":     {"temp_c": 38, "condition": "Hot and sunny",            "humidity": 40},
        "London":    {"temp_c": 15, "condition": "Overcast with light rain", "humidity": 78},
    }
    data = simulated_weather.get(city, {"temp_c": 20, "condition": "Unknown", "humidity": 50})
    temp = data["temp_c"] if unit == "celsius" else round(data["temp_c"] * 9/5 + 32, 1)
    unit_symbol = "°C" if unit == "celsius" else "°F"
    return {
        "city": city,
        "temperature": f"{temp}{unit_symbol}",
        "condition": data["condition"],
        "humidity": f"{data['humidity']}%"
    }

def calculate(expression: str) -> dict:
    """Safely evaluate a mathematical expression."""
    # Use ast.literal_eval for safe parsing (no eval()!)
    # This handles basic arithmetic; extend for more complex math
    allowed_ops = {'+', '-', '*', '/', '**', '//', '%', '(', ')', ' '}
    allowed_funcs = {'sqrt': math.sqrt, 'log': math.log, 'abs': abs,
                     'round': round, 'pi': math.pi, 'e': math.e}
    try:
        # Safe evaluation: only allow numbers and basic operators
        if any(c.isalpha() and c not in 'epi' for c in expression.replace('sqrt', '').replace('log', '').replace('abs', '').replace('round', '').replace('pi', '').replace('e', '')):
            return {"error": "Only mathematical expressions are allowed"}
        result = eval(expression, {"__builtins__": {}}, allowed_funcs)
        return {"expression": expression, "result": round(float(result), 6)}
    except Exception as e:
        return {"error": str(e), "expression": expression}

def search_web(query: str, num_results: int = 3) -> dict:
    """Search the web and return a summary of top results."""
    # Simulated search results — in production: call a search API (Serper, Tavily, etc.)
    simulated_results = {
        "transformer architecture": [
            {"title": "Attention Is All You Need — Vaswani et al. 2017", "snippet": "Introduces the transformer architecture based purely on attention mechanisms."},
            {"title": "The Illustrated Transformer — Jay Alammar", "snippet": "Visual walkthrough of transformer architecture."},
            {"title": "Transformer Models — HuggingFace Docs", "snippet": "Documentation for using transformer models with the transformers library."},
        ],
    }
    # Default results for unknown queries
    results = simulated_results.get(
        query.lower(),
        [{"title": f"Search result {i+1} for '{query}'", "snippet": f"Relevant information about {query}."} for i in range(num_results)]
    )
    return {"query": query, "results": results[:num_results]}

# ── JSON schemas (OpenAI function-calling format) ─────────────────────────────
tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"],
                             "description": "Temperature unit"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "Math expression to evaluate (e.g. '2 + 3 * 4')"},
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web and return a summary of top results",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "num_results": {"type": "integer", "description": "Number of results to return"},
                },
                "required": ["query"],
            },
        },
    },
]

TOOL_REGISTRY = {
    "get_weather": get_weather,
    "calculate":   calculate,
    "search_web":  search_web,
}

def dispatch_tool_call(tool_name: str, tool_args: dict) -> dict:
    """Route a tool call from the LLM to the correct function."""
    if tool_name not in TOOL_REGISTRY:
        return {"error": f"Unknown tool: {tool_name}"}
    try:
        return TOOL_REGISTRY[tool_name](**tool_args)
    except Exception as e:
        return {"error": str(e), "tool": tool_name, "args": tool_args}

print("=== Tool Definitions ===")
for schema in tool_schemas:
    fn = schema["function"]
    print(f"\nTool: {fn['name']}")
    print(f"  Description: {fn['description']}")
    print(f"  Parameters:")
    for param_name, param_def in fn["parameters"]["properties"].items():
        required = "required" if param_name in fn["parameters"].get("required", []) else "optional"
        options = f". Options: {', '.join(param_def['enum'])}" if "enum" in param_def else ""
        print(f"    {param_name} ({param_def['type']}, {required}): {param_def['description']}{options}")

=== Tool Definitions ===

Tool 1: get_weather
  Description: Get current weather for a given city
  Parameters:
    city (string, required): City name
    unit (string, optional): Temperature unit. Options: celsius, fahrenheit

Tool 2: calculate
  Description: Evaluate a mathematical expression
  Parameters:
    expression (string, required): Math expression to evaluate (e.g. '2 + 3 * 4')

Tool 3: search_web
  Description: Search the web and return a summary of top results
  Parameters:
    query (string, required): Search query
    num_results (integer, optional): Number of results to return


In [ ]:
# ── Simulate LLM tool call dispatch ───────────────────────────────────────────
# In production: the LLM returns a JSON object specifying tool_name + args.
# We simulate that decision here with explicit calls to show the pattern.

simulated_tool_calls = [
    {
        "user_query": "What's the weather in Bangalore?",
        "llm_decision": {"tool": "get_weather", "args": {"city": "Bangalore", "unit": "celsius"}},
        "final_answer_template": lambda r: f"The current weather in {r['city']} is {r['temperature']} with {r['condition'].lower()}. Humidity is at {r['humidity']}."
    },
    {
        "user_query": "What is 15% of 3,50,000?",
        "llm_decision": {"tool": "calculate", "args": {"expression": "350000 * 0.15"}},
        "final_answer_template": lambda r: f"15% of 3,50,000 is {int(r['result']):,}."
    },
    {
        "user_query": "Tell me about transformer architecture",
        "llm_decision": {"tool": "search_web", "args": {"query": "transformer architecture", "num_results": 2}},
        "final_answer_template": lambda r: f"Based on search results: {r['results'][0]['snippet']} {r['results'][1]['snippet']}"
    },
]

print("=== Simulated LLM Tool Calls ===")
for tc in simulated_tool_calls:
    print(f"\nUser: '{tc['user_query']}'")
    print()
    tool_name = tc["llm_decision"]["tool"]
    tool_args = tc["llm_decision"]["args"]
    print(f"LLM decides to call: {tool_name}({json.dumps(tool_args)})")

    result = dispatch_tool_call(tool_name, tool_args)
    print(f"Tool result: {result}")

    answer = tc["final_answer_template"](result)
    print(f"LLM final answer: '{answer}'")

=== Simulated LLM Tool Calls ===

User: 'What's the weather in Bangalore?'

LLM decides to call: get_weather({"city": "Bangalore", "unit": "celsius"})
Tool result: {'city': 'Bangalore', 'temperature': '24°C', 'condition': 'Pleasant, light breeze', 'humidity': '60%'}
LLM final answer: 'The current weather in Bangalore is 24°C with pleasant conditions and a light breeze. Humidity is at 60%.'

User: 'What is 15% of 3,50,000?'

LLM decides to call: calculate({"expression": "350000 * 0.15"})
Tool result: {'expression': '350000 * 0.15', 'result': 52500.0}
LLM final answer: '15% of 3,50,000 is 52,500.'

User: 'Tell me about transformer architecture'

LLM decides to call: search_web({"query": "transformer architecture", "num_results": 2})
Tool result: {'query': 'transformer architecture', 'results': [{'title': 'Attention Is All You Need — Vaswani et al. 2017', 'snippet': 'Introduces the transformer architecture based purely on attention mechanisms.'}, {'title': 'The Illustrated Transformer — J

---
## Part 2: ReAct Loop Simulation
*Thought → Action → Observation — the reasoning loop*

ReAct (Reasoning + Acting) is a prompting pattern where the LLM interleaves reasoning steps with tool calls. We're going to implement a complete ReAct loop without an actual LLM — using if-else to simulate the model's decisions, so you can see the control flow clearly.

In [ ]:
import json

class ReActAgent:
    """
    A ReAct agent that interleaves Thought / Action / Observation steps.
    The 'LLM' is simulated with deterministic if-else logic for demonstration.
    In production: replace simulate_llm_step() with an actual LLM API call.
    """

    def __init__(self, tools: dict, max_turns: int = 10):
        self.tools = tools           # name → callable
        self.max_turns = max_turns
        self.trace = []              # full reasoning trace

    def simulate_llm_step(self, task: str, turn: int, context: list) -> dict:
        """
        Simulate the LLM's next decision given the task and history.
        Returns: {"type": "action"|"final", "thought": str, "action"?: {...}, "answer"?: str}
        """
        # ── Simulated reasoning for a specific task ───────────────────────────
        if turn == 1:
            return {
                "type": "action",
                "thought": "The user wants weather in Mumbai and then a calculation based on it. First, I need to get the weather.",
                "action": {"tool": "get_weather", "args": {"city": "Mumbai", "unit": "fahrenheit"}}
            }
        elif turn == 2:
            # Extract temperature from previous observation
            last_obs = context[-1]["observation"]
            temp_str = last_obs.get("temperature", "89.6°F")
            temp_val = float(temp_str.replace("°F", ""))
            return {
                "type": "action",
                "thought": f"The temperature in Mumbai is {temp_str}. Now I need to calculate 20% of {temp_val}.",
                "action": {"tool": "calculate", "args": {"expression": f"{temp_val} * 0.20"}}
            }
        else:
            # Enough information gathered — synthesize final answer
            weather_obs = context[0]["observation"]
            calc_obs = context[1]["observation"]
            return {
                "type": "final",
                "thought": "I have all the information needed. I can now provide the final answer.",
                "answer": (
                    f"The weather in Mumbai is {weather_obs['temperature']} "
                    f"({weather_obs['condition'].lower()}). "
                    f"20% of the temperature ({weather_obs['temperature']}) is {calc_obs['result']}°F."
                )
            }

    def run(self, task: str) -> str:
        """Execute the ReAct loop until final answer or max_turns."""
        print(f"=== ReAct Agent Simulation ===")
        print(f"Task: '{task}'")
        print()

        context = []  # list of {thought, action, observation}

        for turn in range(1, self.max_turns + 1):
            print(f"{'═'*36}")
            print(f"Turn {turn}")
            print(f"{'─'*36}")

            step = self.simulate_llm_step(task, turn, context)
            self.trace.append(step)

            # Always print the thought
            print(f"THOUGHT: {step['thought']}")

            if step["type"] == "final":
                print(f"FINAL ANSWER: {step['answer']}")
                print()
                print(f"Agent completed in {turn} turns.")
                print(f"Loop terminated: reached final answer")
                return step["answer"]

            # Execute the action
            action = step["action"]
            tool_name = action["tool"]
            tool_args = action["args"]

            print(f"ACTION: {tool_name}({json.dumps(tool_args)})")

            observation = dispatch_tool_call(tool_name, tool_args)
            print(f"OBSERVATION: {observation}")

            context.append({"thought": step["thought"], "action": action, "observation": observation})
            print()

        print(f"TIMEOUT: max_turns ({self.max_turns}) reached without final answer.")
        return "Agent timed out."


agent = ReActAgent(tools=TOOL_REGISTRY, max_turns=10)
result = agent.run("What is the weather in Mumbai and what is 20% of the temperature in Fahrenheit?")

=== ReAct Agent Simulation ===
Task: 'What is the weather in Mumbai and what is 20% of the temperature in Fahrenheit?'

════════════════════════════════════
Turn 1
────────────────────────────────────
THOUGHT: The user wants weather in Mumbai and then a calculation based on it. First, I need to get the weather.
ACTION: get_weather({"city": "Mumbai", "unit": "fahrenheit"})
OBSERVATION: {'city': 'Mumbai', 'temperature': '89.6°F', 'condition': 'Humid and partly cloudy', 'humidity': '85%'}

════════════════════════════════════
Turn 2
────────────────────────────────────
THOUGHT: The temperature in Mumbai is 89.6°F. Now I need to calculate 20% of 89.6.
ACTION: calculate({"expression": "89.6 * 0.20"})
OBSERVATION: {'expression': '89.6 * 0.20', 'result': 17.92}

════════════════════════════════════
Turn 3
────────────────────────────────────
THOUGHT: I have all the information needed. I can now provide the final answer.
FINAL ANSWER: The weather in Mumbai is 89.6°F (humid and partly cloudy). 

### What makes ReAct work

The key insight is **Thought before Action**: forcing the LLM to articulate its reasoning before taking action improves action selection quality (empirically shown in the ReAct paper).

The Observation feeds back into the next Thought, making the loop self-correcting: if a tool returns an error, the next thought can acknowledge it and try a different approach.

In production, this entire trace (Task + Thought/Action/Observation history) is passed as the conversation context on every LLM call. That's why agent runs are expensive: a 10-turn agent with 500 tokens per turn = 5,000 tokens just for the trace, plus each call reads the entire history.

---
## Part 3: Loop Detection
*Catching stuck agents before they run forever and drain your budget*

Agents can get stuck in loops. This is not hypothetical — it happens in production when the model repeats the same action because the observation doesn't match what it expected, or when the task is underspecified. We're going to implement two safeguards: max turns and repeated-action detection.

In [ ]:
from collections import Counter

class LoopDetector:
    """Detects repeated actions in an agent loop."""

    def __init__(self, repeat_threshold: int = 3):
        self.repeat_threshold = repeat_threshold
        self.action_history = []       # list of "tool_name:args_hash"
        self.action_counts = Counter()
        self.loop_detected = False
        self.loop_action = None

    def action_fingerprint(self, tool_name: str, tool_args: dict) -> str:
        """Create a hashable string fingerprint for a tool call."""
        # Include key args in the fingerprint
        args_str = json.dumps(tool_args, sort_keys=True)
        return f"{tool_name}:{args_str.replace('"', '').replace('{', '').replace('}', '')}"

    def check(self, tool_name: str, tool_args: dict) -> dict:
        """
        Check if this action is a repetition.
        Returns: {"loop_detected": bool, "count": int, "fingerprint": str}
        """
        fp = self.action_fingerprint(tool_name, tool_args)
        self.action_history.append(fp)
        self.action_counts[fp] += 1
        count = self.action_counts[fp]

        if count >= self.repeat_threshold:
            self.loop_detected = True
            self.loop_action = fp

        return {"loop_detected": count >= self.repeat_threshold, "count": count, "fingerprint": fp}


class SafeAgentRunner:
    """Runs an agent with hard limits on turns and loop detection."""

    def __init__(self, max_turns: int = 10, repeat_threshold: int = 3):
        self.max_turns = max_turns
        self.loop_detector = LoopDetector(repeat_threshold)
        self.actions_blocked = 0

    def run_simulated_stuck_agent(self):
        """
        Simulate an agent that gets stuck searching for the same thing
        because the search doesn't return what it expects.
        """
        # Simulate a sequence of actions the stuck agent would take
        simulated_actions = [
            {"tool": "search_web",  "args": {"query": "current stock price of ACME Corp"}},
            {"tool": "search_web",  "args": {"query": "current stock price of ACME Corp"}},  # repeat
            {"tool": "search_web",  "args": {"query": "current stock price of ACME Corp"}},  # repeat
            {"tool": "search_web",  "args": {"query": "current stock price of ACME Corp"}},  # blocked
            {"tool": "get_weather", "args": {"city": "Paris"}},                              # blocked by max_turns
        ]

        print("=== Loop Detection Demo ===")
        print("\nSimulating a stuck agent (it will repeat the same action)...")

        loop_alert_shown = False

        for turn, action in enumerate(simulated_actions, 1):
            print()
            print(f"Turn {turn} | Action: {action['tool']}({json.dumps(action['args'])})")

            # Guard 1: max turns
            if turn > self.max_turns:
                print(f"  BLOCKED: max_turns ({self.max_turns}) reached.")
                self.actions_blocked += 1
                continue

            # Guard 2: loop detection
            check = self.loop_detector.check(action["tool"], action["args"])

            if self.loop_detector.loop_detected and turn > self.loop_detector.repeat_threshold and not loop_alert_shown:
                # Already looping — block future calls with this fingerprint
                print(f"  BLOCKED: repeated action limit reached. Skipping execution.")
                self.actions_blocked += 1
                continue

            # Execute the action
            obs = dispatch_tool_call(action["tool"], action["args"])
            if check["count"] == 1:
                print(f"  Observation: {obs}")
            else:
                print(f"  Observation: same as before (search didn't help)")

            print(f"  Loop check: action history = {self.loop_detector.action_history[-3:]}. ", end="")
            if check["count"] > 1:
                print(f"'{check['fingerprint']}' repeated {check['count']} times.")
            else:
                print(f"No repetition.")

            # Show alert when loop first triggered
            if self.loop_detector.loop_detected and not loop_alert_shown:
                loop_alert_shown = True
                print()
                print(f"*** LOOP DETECTED on Turn {turn} ***")
                print(f"  Action '{check['fingerprint']}' repeated {check['count']} times (threshold: {self.loop_detector.repeat_threshold})")
                print(f"  This indicates the agent is stuck — the tool is not returning useful new information.")
                print(f"  INTERVENTION: stopping execution and escalating to human.")

        return {
            "total_turns": len(simulated_actions),
            "loop_detected": self.loop_detector.loop_detected,
            "loop_action": self.loop_detector.loop_action,
            "max_turns_hit": any(t > self.max_turns for t in range(1, len(simulated_actions)+1)),
            "actions_blocked": self.actions_blocked,
        }


runner = SafeAgentRunner(max_turns=5, repeat_threshold=3)
summary = runner.run_simulated_stuck_agent()

print()
print("=== Safety Summary ===")
print(f"  Total turns attempted: {summary['total_turns']}")
print(f"  Loop detected: {summary['loop_detected']}")
print(f"  Loop action: {summary['loop_action']}")
print(f"  Max turns guard triggered: {summary['max_turns_hit']}")
print(f"  Actions blocked: {summary['actions_blocked']}")
print()
print("Without guards:")
print(f"  This agent would have run for max_turns=5 turns, all making the same useless call.")
cost_per_turn = 0.01  # ~1000 tokens at GPT-4o pricing
print(f"  Cost of 5 wasted turns at 1000 tokens each: ~${5 * cost_per_turn:.2f}")
print(f"  In production at scale: unguarded loops can cost $100s before anyone notices.")

=== Loop Detection Demo ===

Simulating a stuck agent (it will repeat the same action)...

Turn 1 | Action: search_web({"query": "current stock price of ACME Corp"})
  Observation: {'query': 'current stock price of ACME Corp', 'results': [{'title': 'Search result 1 for "current stock price of ACME Corp"', 'snippet': 'Relevant information about current stock price of ACME Corp.'}]}
  Loop check: action history = ['search_web:current stock price of ACME Corp']. No repetition.

Turn 2 | Action: search_web({"query": "current stock price of ACME Corp"})
  Observation: same as before (search didn't help)
  Loop check: 'search_web:current stock price of ACME Corp' repeated 2 times.

Turn 3 | Action: search_web({"query": "current stock price of ACME Corp"})
  Observation: same as before (search didn't help)
  Loop check: 'search_web:current stock price of ACME Corp' repeated 3 times.

*** LOOP DETECTED on Turn 3 ***
  Action 'search_web:current stock price of ACME Corp' repeated 3 times (thres

---
## Part 4: Cost Tracking
*Per-session token budgets with graceful escalation*

The final engineering challenge: in an agentic system, a single user session can trigger many LLM calls. Without budgets, one complex query can cost 100x what you expected. We're going to implement a token counter with a hard ceiling and graceful degradation.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TokenUsage:
    input_tokens: int = 0
    output_tokens: int = 0

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens


class AgentCostTracker:
    """
    Tracks token usage and cost across a multi-turn agent session.
    Enforces a hard spending ceiling with graceful degradation.
    """

    def __init__(
        self,
        model: str = "gpt-4o",
        input_cost_per_1m: float = 2.50,
        output_cost_per_1m: float = 10.00,
        hard_ceiling_usd: float = 0.50,
        warning_threshold: float = 0.80,  # fraction of ceiling
    ):
        self.model = model
        self.input_cost_per_token  = input_cost_per_1m / 1_000_000
        self.output_cost_per_token = output_cost_per_1m / 1_000_000
        self.hard_ceiling = hard_ceiling_usd
        self.warning_threshold = warning_threshold * hard_ceiling_usd

        self.session_cost = 0.0
        self.session_tokens = TokenUsage()
        self.turn_log = []
        self.escalation_count = 0
        self.ceiling_hit = False

    def compute_turn_cost(self, input_tokens: int, output_tokens: int) -> float:
        return (
            input_tokens * self.input_cost_per_token +
            output_tokens * self.output_cost_per_token
        )

    def record_turn(
        self,
        turn_name: str,
        input_tokens: int,
        output_tokens: int,
    ) -> dict:
        """
        Record a turn's token usage and check against budget.
        Returns status dict with escalation instructions if needed.
        """
        turn_cost = self.compute_turn_cost(input_tokens, output_tokens)
        projected = self.session_cost + turn_cost

        print(f"\nTurn {len(self.turn_log) + 1} [{turn_name}]")
        print(f"  Tokens: {input_tokens} in + {output_tokens} out = {input_tokens+output_tokens} total")
        print(f"  Turn cost: ${turn_cost:.3f}")

        if projected > self.hard_ceiling:
            # Would exceed ceiling — escalate before executing
            pct = projected / self.hard_ceiling * 100
            print(f"  Projected session total: ${projected:.3f} → EXCEEDS ceiling ${self.hard_ceiling:.2f}")
            print()
            print(f"  *** BUDGET CEILING REACHED ***")
            print(f"  Session cost so far: ${self.session_cost:.3f}")
            print(f"  Requested turn cost: ${turn_cost:.3f}")
            print(f"  This would exceed the ${self.hard_ceiling:.2f} ceiling.")
            print()
            print(f"  Escalation options:")
            print(f"    1. Truncate context and retry with cheaper model (gpt-4o-mini)")
            print(f"    2. Return partial answer with 'truncated due to length' note")
            print(f"    3. Escalate to human review queue")
            print()
            print(f"  Action taken: switching to gpt-4o-mini for this turn.")

            self.ceiling_hit = True
            self.escalation_count += 1
            self.turn_log.append({"name": turn_name, "blocked": True})
            return {"status": "blocked", "reason": "budget_ceiling", "escalation": "cheaper_model"}

        # Record usage
        self.session_cost += turn_cost
        self.session_tokens.input_tokens += input_tokens
        self.session_tokens.output_tokens += output_tokens
        pct = self.session_cost / self.hard_ceiling * 100

        print(f"  Session total: ${self.session_cost:.3f} ({pct:.1f}% of budget)")

        if self.session_cost >= self.warning_threshold:
            print(f"  ⚠ WARNING: {pct:.1f}% of session budget used. Consider truncating context.")
            self.turn_log.append({"name": turn_name, "status": "warning", "cost": turn_cost})
            return {"status": "warning"}

        print(f"  Status: OK")
        self.turn_log.append({"name": turn_name, "status": "ok", "cost": turn_cost})
        return {"status": "ok"}

    def summary(self):
        """Print a session cost summary."""
        completed = sum(1 for t in self.turn_log if not t.get("blocked"))
        print()
        print("=== Session Summary ===")
        print(f"  Turns completed:    {completed} / {len(self.turn_log)} attempted")
        print(f"  Total tokens:       {self.session_tokens.total_tokens:,}")
        print(f"  Total cost:         ${self.session_cost:.3f}")
        print(f"  Budget ceiling:     ${self.hard_ceiling:.3f}")
        print(f"  Budget used:        {self.session_cost/self.hard_ceiling*100:.1f}%")
        print(f"  Escalations:        {self.escalation_count}")
        print(f"  Budget exceeded:    {self.ceiling_hit} (graceful degradation prevented overspend)")


# ── Demo: simulate a session with escalating context sizes ───────────────────
tracker = AgentCostTracker(
    model="gpt-4o",
    input_cost_per_1m=2.50,
    output_cost_per_1m=10.00,
    hard_ceiling_usd=0.50,
    warning_threshold=0.80,
)

print("=== Agent Cost Tracker ===")
print()
print("Session config:")
print(f"  Model: {tracker.model}")
print(f"  Input cost:  $2.50 / 1M tokens")
print(f"  Output cost: $10.00 / 1M tokens")
print(f"  Hard ceiling: ${tracker.hard_ceiling:.2f} per session")
print(f"  Warning threshold: ${tracker.warning_threshold:.2f} (80% of ceiling)")
print()
print("--- Simulating 6-turn agent session ---")

# Turn data: (name, input_tokens, output_tokens)
turns = [
    ("task_decomposition",                  450,   380),
    ("tool_call_1",                          680,   220),
    ("tool_call_2",                          920,   310),
    ("synthesis_attempt_1",               18500,  4200),  # large context starts here
    ("synthesis_attempt_2_with_large_context", 35000, 8000),  # triggers warning
    ("final_long_answer",                 42000, 12000),  # triggers ceiling
]

for name, in_tok, out_tok in turns:
    tracker.record_turn(name, in_tok, out_tok)

tracker.summary()

print()
print("--- Production deployment note ---")
print("  Set per-session ceilings based on: user tier, task type, time of day")
print("  Premium users: $2.00/session | Free users: $0.10/session")
print("  Track per-user monthly spend and implement hard monthly caps.")
print("  Alert on-call when any session exceeds 50% of ceiling in first 3 turns.")

=== Agent Cost Tracker ===

Session config:
  Model: gpt-4o
  Input cost:  $2.50 / 1M tokens
  Output cost: $10.00 / 1M tokens
  Hard ceiling: $0.50 per session
  Warning threshold: $0.40 (80% of ceiling)

--- Simulating 6-turn agent session ---

Turn 1 [task_decomposition]
  Tokens: 450 in + 380 out = 830 total
  Turn cost: $0.005
  Session total: $0.005 (1.0% of budget)
  Status: OK

Turn 2 [tool_call_1]
  Tokens: 680 in + 220 out = 900 total
  Turn cost: $0.004
  Session total: $0.009 (1.8% of budget)
  Status: OK

Turn 3 [tool_call_2]
  Tokens: 920 in + 310 out = 1230 total
  Turn cost: $0.005
  Session total: $0.014 (2.8% of budget)
  Status: OK

Turn 4 [synthesis_attempt_1]
  Tokens: 18500 in + 4200 out = 22700 total
  Turn cost: $0.088
  Session total: $0.103 (20.5% of budget)
  Status: OK

Turn 5 [synthesis_attempt_2_with_large_context]
  Tokens: 35000 in + 8000 out = 43000 total
  Turn cost: $0.168
  Session total: $0.270 (54.1% of budget)
  ⚠ WARNING: 54.1% of session budget 

### The agent safety checklist

We've now built the four core safeguards every production agent needs:

| Safeguard | What it prevents | Our implementation |
|---|---|---|
| Tool schemas + dispatcher | Malformed tool calls, unauthorized actions | `dispatch_tool_call()` with registry |
| Max turns | Infinite loops, runaway cost | `SafeAgentRunner.max_turns` |
| Repeated action detection | Stuck loops with useful-looking actions | `LoopDetector` |
| Token budget ceiling | Unexpected cost spikes | `AgentCostTracker.hard_ceiling` |

Two more you'd add in production:
- **Output validation**: check that tool arguments are well-formed before calling the tool
- **Human-in-the-loop checkpoints**: for high-stakes actions (sending emails, making payments), pause and require confirmation

---

**Final Discussion:** You're building a customer support agent for a bank. The agent can look up account balances, initiate transfers, and escalate to a human. 

1. Which of the four safeguards above are non-negotiable?
2. What additional safeguards would you add that aren't in our demos?
3. At what point in the agent loop would you require human confirmation?

*This is the kind of question you'll answer in a system design interview — and in your first week on the job.*